# 06. Pose 좌표 → LLM → ROS / 105분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

LLM은 좌표를 새로 발명하는 대신 관측에 존재하는 관절을 선택합니다. 최종 좌표는 검출기의 원래 값을 사용합니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab06_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![Pose 좌표 → LLM → ROS의 입력·처리·출력](../교재그림/guided-vla/day4-06.svg)

pose_prompt는 선택된 사람의 bbox와 전체 관절을 텍스트로 전달합니다. validate_pose_decision은 동일 frame, 요청한 관절, 관절 신뢰도와 좌표 근거를 검사합니다. pose_motion은 승인된 결과에도 시간과 turtlesim pose 검사를 반복합니다. 사람 미검출 시 모델 요청 없이 정지합니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![4일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day4-system.svg)

터미널 A/B/C는 실행·관찰 역할이고 ROS 노드와 HTTP 서비스는 실제 처리 주체입니다. 02는 파일 요청만, 05는 독립 YOLO 실행만 다루므로 매 단원에서 모든 상자를 동시에 실행하지 않습니다. 아래 단원별 도식과 함께 현재 경로를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 이미지 VLM과 좌표 LLM의 차이

- 이번 요청은 이미지 없이 전체 관절 JSON을 보냅니다.
- LLM은 지시한 관절을 고르고 원래 좌표를 답합니다.
- left/right는 사람의 해부학적 좌우입니다.
- 검출된 숫자도 누락·가림·오차를 가질 수 있습니다.

02·03에서 VLM은 이미지 픽셀을 보고 손 모양을 판단했습니다. 이번에는 YOLO가 먼저 이미지에서 사람과 관절을 추출하고 LLM은 그 구조화된 텍스트를 읽습니다. 같은 모델 tag를 사용하더라도 payload에 images가 없으므로 이 요청의 정보 경로는 텍스트입니다. 검출기가 놓친 시각 정보를 LLM이 직접 다시 보는 구조가 아닙니다.

pose_prompt는 선택된 한 사람의 bbox와 17관절을 모두 포함하고 instruction과 target_keypoint를 명시합니다. 관절 이름의 left/right는 사람 자신의 좌우이며 화면 왼쪽·오른쪽과 다를 수 있습니다. 거울처럼 표시한다고 관절 이름을 임의로 바꾸면 안 됩니다.

관절 선택은 교육용으로 제한된 과제입니다. nose를 지시하면 허용할 target도 nose이고, 좌표는 관측에 존재해야 합니다. LLM이 새로운 위치를 추정하거나 낮은 신뢰도의 관절을 그럴듯하게 보완해 움직이게 하지 않습니다. 자연어 인터페이스와 실제 제어 근거의 경계를 보는 실습입니다.


### 2. 실행 아키텍처: YOLO·LLM·제어기 경계

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day4-06.svg)

- 카메라 → pose worker → 유효한 최대 면적 사람
- 전체 관절 텍스트 → Ollama → raw JSON
- validate_pose_decision → 원본 관절 좌표로 승인
- 20Hz pose_motion → cmd_vel; /pose/*로 관찰

PoseFollow.process는 BGR 영상에서 YOLO pose를 실행하고 people_from_yolo와 pose_observation으로 관측을 만듭니다. 유효한 사람이 없으면 LLM을 호출하지 않고 같은 frame의 STOP을 만듭니다. 사람이 있으면 전체 관절 텍스트를 보내고 raw 응답을 원본 관측과 대조합니다.

검증된 decision과 observation, issued_ms, control_image는 함께 timer로 전달됩니다. timer는 손 추종 때와 마찬가지로 촬영 age·TTL·Pose age를 매번 검사합니다. /pose/perception은 관측, /pose/decision은 LLM 검증 결과, /pose/action은 최종 속도와 이유, /pose/image_overlay는 그 근거 화면입니다. 마지막 이동 토픽은 같은 /turtle1/cmd_vel입니다.

log의 yolo_ms와 llm_ms는 경로의 서로 다른 구간입니다. 사람이 없어 요청을 생략하면 llm_ms는 0입니다. 이를 모델이 즉시 추론했다고 해석하지 않습니다. latency_ms는 process 전체 경과 시간에 대응하므로 구성 요소와 범위를 확인해 읽습니다.


### 3. 코드 추적: 형식에서 좌표 근거까지

- parse: STOP이면 keypoint·center는 null이어야 합니다.
- validate: frame·요청 관절·신뢰도·좌표 오차를 대조합니다.
- 전송은 소수4자리, 검증은 원래 정밀도의 관측입니다.
- 승인 뒤에도 반환 center는 검출기 원본 좌표입니다.

parse_pose_decision은 JSON 중복 key·NaN·필드 집합·허용 관절 이름·0~1 좌표를 검사합니다. STOP인데 좌표가 남아 있으면 의미가 모호하므로 거부합니다. 형식이 맞은 FOLLOW라도 validate_pose_decision에서 frame_id가 관측과 같은지, expected_keypoint인지, 해당 관절 신뢰도가 기준 이상인지 확인합니다.

compact_pose_measurements는 전송용 복사본의 숫자를 소수 네 자리로 줄입니다. 원본 observation을 반올림해서 덮어쓰는 것이 아닙니다. validator의 tolerance=0.002는 각 좌표 축에서 작은 전송 반올림 차이를 허용하는 기준입니다. 거리 전체를 뜻하거나 화면의 넓은 영역을 대충 허용하는 값이 아닙니다.

검증이 통과하면 LLM이 답한 center 대신 관측의 원래 xy를 반환합니다. 이어서 pose_motion이 다시 사람·관절·시간·Pose를 확인한 뒤 목표와 속도를 만듭니다. 검증 실패인 None을 규칙 FOLLOW로 대체하면 LLM이 잘못된 값을 냈다는 사실이 사라지므로 이 구현은 그런 fallback을 하지 않습니다.


### 4. 코·손목 비교와 확장 시 변경 지점

- nose → left_wrist 변경은 지시와 검증 target을 함께 변경
- 가려진 손목은 좌표 유지보다 정지 근거 확인
- 선택 사람 변경·관절 변경·미러링을 구분
- 새 관절 체계는 prompt·schema·adapter·검증을 함께 수정

코와 왼쪽 손목은 움직임 크기와 가림 빈도가 다릅니다. 같은 사람에서 target만 바꿔 목표와 거북이 동작을 비교하면 입력 표현과 제어기는 유지하면서 목표 선택의 영향을 볼 수 있습니다. 손목이 가려졌을 때 이전 좌표가 계속 쓰이는지, 새 관측에서 거부되는지, 아직 이전 승인 TTL 구간인지 로그로 구분합니다.

target_keypoint와 instruction을 서로 모순되게 두면 모델의 답이 자연어에는 맞지만 validator에서 거부될 수 있습니다. launch에서 두 값을 함께 바꾸고 prompt 및 expected_keypoint가 어떻게 연결되는지 확인합니다. tolerance를 크게 늘려 거부를 없애는 것은 관측 근거가 없는 좌표까지 허용하는 다른 정책입니다.

관절 체계가 다른 모델로 교체할 때는 COCO_KEYPOINTS, adapter, schema, prompt, validator와 overlay를 함께 검토합니다. 이번 실습을 마치면 이미지 의미를 모델에 맡기든 숫자 선택을 맡기든 최종 속도를 만드는 별도 제어와 시간 검사가 필요하다는 공통 구조를 이해할 수 있습니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [ros_ws/src/nepes_vla_demo/nepes_vla_core/pose_contracts.py](ros_ws/src/nepes_vla_demo/nepes_vla_core/pose_contracts.py) |
| 바꾸어 볼 조건 | `target_keypoint:=nose / left_wrist` |
| 예상되는 차이 | 코와 왼쪽 손목을 각각 승인한 뒤 잘못된 frame, 존재하지 않는 좌표, 낮은 관절 신뢰도를 거부하는 것을 봅니다. 실제 실행은 코와 손목 조건을 차례로 비교합니다. |
| 직접 구현·수정할 때 | target_keypoint와 자유 지시 instruction의 관절을 일치시킵니다. frame·관절 confidence·좌표 일치·시간 검사를 유지합니다. tolerance는 작은 반올림 허용치이지 모델의 임의 좌표를 승인하는 범위가 아닙니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 3. 준비 — 실제 실행 코드와 같은 함수를 사용합니다

아래 CPU 예제는 카메라·모델·ROS 없이 자료의 흐름을 분리해 보여 줍니다. 통과가 실제 인식 성공이나 장비 성능을 뜻하지는 않습니다.


In [ ]:
from pathlib import Path
import sys, json, inspect, uuid
from datetime import datetime, timezone
from IPython.display import display, Markdown, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'09.Pseudo-VLA_v2').is_dir())
LESSON = ROOT/'09.Pseudo-VLA_v2'
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
# 매 단원 새 커널: 다른 일차의 common 모듈이 남아 있지 않게 합니다.
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = ROOT/'Data/outputs/09.Pseudo-VLA_v2/guided'/RUN

# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
def show_rows(rows):
    """조건·결과를 한 표에서 읽습니다. 수강생이 답안 dict를 채우지 않습니다."""
    if not rows:
        print('표시할 결과 없음')
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    def value(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    lines = [' | '.join(keys), ' | '.join(['---']*len(keys))]
    lines += [' | '.join(value(row.get(key, '')) for key in keys) for row in rows]
    display(Markdown('\n'.join(lines)))

# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
def save_demo(name, rows):
    """합성 예제임을 파일에도 명시합니다. 모델 측정값이나 수강생 답안이 아닙니다."""
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT/(name+'.json')
    path.write_text(json.dumps({'kind':'synthetic-guided-demo', 'model_measured':False,
                               'rows':rows}, ensure_ascii=False, indent=2)+'\n')
    print('자동 저장:', path)
print('실행 폴더:', LESSON)


### 완성 코드 읽기

아래 셀은 실행에 쓰는 함수 본문을 그대로 표시합니다. **입력 검사 → 조건 분기 → 결과 반환**을 순서대로 읽습니다. 노트북에 복사한 별도 정답과 실제 동작이 어긋나지 않도록 같은 소스를 조회합니다.


In [ ]:
from common.pose_contracts import pose_prompt, validate_pose_decision, pose_motion
# 함수의 실제 파일과 주석을 함께 표시합니다. 이 셀은 소스를 수정하지 않습니다.
for function in (pose_prompt, validate_pose_decision, pose_motion,):
    print(inspect.getsourcefile(function))
    display(Code(inspect.getsource(function), language='python'))


### 한 줄씩 읽는 처리 과정

pose_prompt는 선택된 사람의 bbox와 전체 관절을 텍스트로 전달합니다. validate_pose_decision은 동일 frame, 요청한 관절, 관절 신뢰도와 좌표 근거를 검사합니다. pose_motion은 승인된 결과에도 시간과 turtlesim pose 검사를 반복합니다. 사람 미검출 시 모델 요청 없이 정지합니다.

target_keypoint와 자유 지시 instruction의 관절을 일치시킵니다. frame·관절 confidence·좌표 일치·시간 검사를 유지합니다. tolerance는 작은 반올림 허용치이지 모델의 임의 좌표를 승인하는 범위가 아닙니다.


## 4. 조건을 바꾸어 실행하고 표로 비교하기

코와 왼쪽 손목을 각각 승인한 뒤 잘못된 frame, 존재하지 않는 좌표, 낮은 관절 신뢰도를 거부하는 것을 봅니다. 실제 실행은 코와 손목 조건을 차례로 비교합니다.

아래 입력은 경계 조건을 설명하기 위한 고정 예제입니다. 주석을 읽고 셀 전체를 실행하면 비교표와 검사 결과가 자동으로 나옵니다.


In [ ]:
from common.pose_contracts import pose_observation, pose_prompt, validate_pose_decision
person = dict(bbox=[.1,.1,.9,.9], confidence=.9,
              keypoints={'nose': dict(xy=[.5,.3], confidence=.9),
                         'left_wrist': dict(xy=[.2,.6], confidence=.8)})
obs = pose_observation([person], 'pose-demo', 1000)
print('LLM에 전달하는 텍스트:', pose_prompt(obs, target_keypoint='nose'))
rows = []
for keypoint in ('nose', 'left_wrist'):
    # 여기서는 응답 검증을 분리해 설명합니다. 실제 LLM 응답은 아래 ROS 실습에서 받습니다.
    response = dict(command='FOLLOW', frame_id='pose-demo', keypoint=keypoint,
                    center=person['keypoints'][keypoint]['xy'], reason='requested joint')
    for case, changes in [('정상', {}), ('다른 frame', {'frame_id': 'other'}),
                          ('근거 없는 좌표', {'center': [.99, .99]})]:
        validated = validate_pose_decision(json.dumps({**response, **changes}), obs,
                                           expected_keypoint=keypoint)
        rows.append(dict(keypoint=keypoint, case=case, accepted=validated is not None, decision=validated))
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert [r['accepted'] for r in rows] == [True, False, False] * 2
# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
save_demo('pose-approval', rows)


## 5. 실제 입력 실행 — 화면과 로그 연결

본수업은 로컬 카메라 화면을 함께 보며 역할을 교대합니다. 아래의 A/B/C는 터미널 역할입니다. 카메라와 제어기는 각각 하나만 실행합니다. 실제 실행의 지연과 인식 결과는 위 합성 예제와 별도로 확인합니다.


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


01–04와 같은 로컬 실습 컨테이너에서 실행합니다. 05의 `live_yolo.py` 웹캠 실행과 기존 손 추종 제어기는 먼저 종료합니다. **새 터미널은 첫 줄 `bash`부터 전체 블록** 을 복사합니다. 로컬·원격 모두 오프라인 수기 서명 명부의 본인 번호를 ROS_DOMAIN_ID로 사용합니다. 예시 7을 본인 번호로 바꾸고 본인 A·B·C 터미널에 같은 값을 적용합니다. 공동 통신 실습에서만 강사가 지정한 동일 명부 번호를 사용합니다. 이미 설정한 같은 터미널에서 이어 실행할 때는 아래에 표시한 재실행 launch만 사용합니다.

**처음 준비 또는 package not found일 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS_v2/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA_v2/scripts/build_workspace.sh
```

`yolo26n-pose.pt`와 공용 Ollama의 `gemma4:e2b-it-qat`가 준비되어 있어야 합니다. 같은 모델을 사용하더라도 이번 요청에는 `images`가 없고 좌표 JSON 텍스트만 들어갑니다. 모델 요청 실패를 정답 응답으로 대신하지 않습니다.

**A — 카메라 입력:** 기존 A가 살아 있으면 추가 실행하지 않습니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

**B — 가장 큰 사람의 코 추종:** 한 번의 launch로 pose·LLM 제어기, turtlesim, overlay 뷰어가 실행됩니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 launch nepes_vla_demo pose_follow.launch.py \
  pose_model:=/workspace/Data/models/yolo26n-pose.pt \
  llm_model:=gemma4:e2b-it-qat \
  target_keypoint:=nose \
  instruction:="Follow the nose of this person. Stop if the nose is invalid." \
  ttl_ms:=400 \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA_v2/06-pose-nose.jsonl
```

**C — 최종 행동:** 새 터미널에서 전체 블록을 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS_v2/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA_v2/install/setup.bash
export ROS_DOMAIN_ID=7 # 7을 본인 수기 서명 명부 번호로 변경
ros2 topic echo /pose/action
```

C의 echo를 Ctrl+C로 종료한 뒤 **같은 C에서** 입력과 판단을 하나씩 봅니다.
```bash
ros2 topic echo /pose/perception
```
위 echo를 Ctrl+C로 종료한 **같은 C에서**:
```bash
ros2 topic echo /pose/decision
```

**화면 읽기:** `/pose/image_overlay`에는 선택된 사람의 박스·관절과 선택한 추종 점이 표시됩니다. 초록은 전진·회전 중, 빨강은 유효한 좌표가 있지만 정지 중입니다. 유효 좌표가 없으면 `NO TARGET`입니다. frame·승인 경과시간·TTL·정지 이유를 함께 읽습니다. 추론에 사용한 영상과 점을 짝지으므로 LLM을 기다리는 동안 화면도 지난 관측을 유지할 수 있습니다. 카메라 원본의 최신 화면과 판단에 사용한 영상은 다를 수 있습니다.

**움직임:** 코를 영상 오른쪽/위로 이동하면 거북이 목표도 오른쪽/위로 이동합니다. 목표와 방향 차이가 크면 먼저 회전하고, 목표에 도착하면 멈춥니다. 가장 큰 사람이 바뀌면 목표 인물도 바뀝니다. 그 사람의 코가 가려져 유효하지 않으면 STOP입니다. 이번 경로는 손가락을 보지 않으므로 주먹을 쥐는 동작은 정지 명령이 아닙니다.

**종료:** A를 멈춘 뒤 C에서 시간 제한에 의한 STOP을 확인하고 B·C를 종료합니다. 다음 조건을 비교할 때는 A를 유지하고 B만 Ctrl+C로 종료합니다. GUI 없는 자습은 B의 launch 앞에 `QT_QPA_PLATFORM=offscreen`, 인자에 `with_image_view:=false`를 사용합니다.


## 6. 결과 해설과 다음 단계

LLM은 좌표를 새로 발명하는 대신 관측에 존재하는 관절을 선택합니다. 최종 좌표는 검출기의 원래 값을 사용합니다.

코와 왼쪽 손목을 각각 승인한 뒤 잘못된 frame, 존재하지 않는 좌표, 낮은 관절 신뢰도를 거부하는 것을 봅니다. 실제 실행은 코와 손목 조건을 차례로 비교합니다.

규칙·이미지 VLM·좌표 LLM에서 입력 표현과 판단 비용은 달라도 최종 제어와 정지 검사가 필요하다는 점으로 마무리합니다.

비교표와 실제 화면·로그를 강사 설명과 대조합니다. 별도 답안 작성은 없으며, 자동 저장 파일과 실행 결과로 진행 상태를 확인합니다.
